In [1]:
import torch
import torch.nn as nn
class reshape(nn.Module):
    '''
    reshapes the 3-order tensor into 6-order tensor

    ----------
    split : list
        split indices to be applied to each mode of the 3-order tensor

    map_type : int
        based on attached - 1 or compressed - 2 splitting method

    device : str
        operation device, default value is cpu


    inputs a 3-order torch.tensor

    returns a 6-order torch.tensor
    '''
    def __init__(self, split, map_type=1, device='cpu'):
        super(reshape, self).__init__()

        self.split = split
        self.map_type = map_type
        self.device = device

    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []

        if self.map_type == 1:
            # Approach 1 : Attached
            C_indices, H_indices, W_indices = [
                [sum(dim // self.split[i] for _ in range(j)) for j in range(self.split[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]

            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1],
                                           H_indices[j]:H_indices[j+1],
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            # Approach 2 : Compressed
            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            C_stride_indices = torch.arange(i, C, self.split[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.split[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.split[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)

        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.split[0] * self.split[1] * self.split[2])
        result = torch.cat(chunks).view(
            batch_size, self.split[0], self.split[1], self.split[2],
            *chunks[0].shape[1:])
        return result

    def forward(self, x):
        chunks = self.split_into_chunks(x)
        output = self.stack_chunks_to_form_tensor(chunks)
        return output

In [2]:
import torch
import torch.nn as nn
import tensorly as tl
from tltorch import TRL, TCL
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from einops import rearrange

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 8

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [5]:
def topk_accuracy(outputs, targets, topk=(1,)):
    '''
    calculates top-k accuracy

    ----------
    outputs : torch.tensor

    targets : torch.tensor

    topk  : tuple
        calculates top k accuracy given outpurs and targets


    reutrns a python dictionary of top i <= k accuracies

    '''
    maxk = max(topk)
    _, topk_indices = torch.topk(input=outputs, k=maxk, dim=1, largest=True, sorted=True)
    correct = topk_indices.eq(targets.view(-1, 1).expand_as(topk_indices))
    accuracies = {}
    for k in topk:
        correct_k = correct[:,:k].float().sum()
        accuracies[k] = {'correct':correct_k, 'accuracy': (correct_k / outputs.shape[0]) * 100.0}
    return accuracies

In [6]:
def cp(module):
  return sum(p.numel() for p in module.parameters())

In [7]:
def print_gpu_memory_usage(stage):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    string = f'{stage} - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB'
    print(string)
    return string


In [8]:
def append_to_file(file_name, text):
    with open(file_name, 'a') as file:
        file.write(text + '\n')

# FC layers

In [9]:
class CNN1(nn.Module):
    def __init__(self):
        super(CNN1, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8, 256, bias = False)
        self.fc2 = nn.Linear(256,10, bias = False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model1 = CNN1().to(device)


In [10]:
classifier1 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256, bias = False),
    nn.Linear(256, 10, bias = False)
)

print(cp(classifier1))
append_to_file(file_name='TRL_report.txt', text=f'FC classifier # parameters {cp(classifier1)}')

1051136


In [11]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters())

In [12]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model1.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model1.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [13]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TRL_report.txt', text=f'FC took {end_time - start_time} time')
append_to_file(file_name='TRL_report.txt', text=f'FC had {string}')
append_to_file(file_name='TRL_report.txt', text=f'FC last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

Train epoch 1: top1=0.551539957523346%, top2=0.7475799918174744%, top3=0.8440600037574768%, top4=0.9040600061416626%, top5=0.941279947757721%, loss=0.1556651172506809, time=7.690049648284912s
Test epoch 1: top1=0.6491000056266785%, top2=0.8294000029563904%, top3=0.9039999842643738%, top4=0.945099949836731%, top5=0.9697999954223633%, loss=0.12235671595931052, time=0.9142193794250488s
Memory Usage  - Allocated: 32.59 MB, Reserved: 68.00 MB
Train epoch 2: top1=0.6887999773025513%, top2=0.8490399718284607%, top3=0.9187600016593933%, top4=0.9549199938774109%, top5=0.9759799838066101%, loss=0.11058364663720131, time=7.188716411590576s
Test epoch 2: top1=0.6784999966621399%, top2=0.8434000015258789%, top3=0.9103999733924866%, top4=0.9495999813079834%, top5=0.9731999635696411%, loss=0.11532022068221122, time=0.9211857318878174s
Train epoch 3: top1=0.7477399706840515%, top2=0.8906999826431274%, top3=0.9454399943351746%, top4=0.9723599553108215%, top5=0.9861599802970886%,

In [14]:
append_to_file(file_name='TRL_report.txt', text=f'########################################')

# TRL from Tensorly

In [15]:
class CNN2(nn.Module):
    def __init__(self):
        super(CNN2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.trl = TRL(input_shape = (64,2,2), output_shape = (10), factorization='tucker', rank=(10,1,1,10))
        # self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = x.reshape(x.size(0), 64,2,2)
        x = self.trl(x)
        return x


model2 = CNN2().to(device)


In [16]:
classifier2 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256),
    TRL(input_shape = (64,2,2), output_shape = (10), factorization='tucker', rank=(10,1,1,10))
)

print(cp(classifier2))
append_to_file(file_name='TRL_report.txt', text=f'TRL tensorly classifier # parameters {cp(classifier2)}')

1049676


In [17]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters())

In [18]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model2.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model2.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [19]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TRL_report.txt', text=f'TRL tensorly took {end_time - start_time} time')
append_to_file(file_name='TRL_report.txt', text=f'TRL tensorly had {string}')
append_to_file(file_name='TRL_report.txt', text=f'TRL tensorly last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

Train epoch 1: top1=0.5082600116729736%, top2=0.7054600119590759%, top3=0.8076199889183044%, top4=0.8735599517822266%, top5=0.9182199835777283%, loss=0.17170075670450927, time=9.372107028961182s
Test epoch 1: top1=0.6011999845504761%, top2=0.7793999910354614%, top3=0.8672999739646912%, top4=0.9202999472618103%, top5=0.9494999647140503%, loss=0.14359176743626595, time=1.088932752609253s
Memory Usage  - Allocated: 40.74 MB, Reserved: 68.00 MB
Train epoch 2: top1=0.6543200016021729%, top2=0.825719952583313%, top3=0.9006199836730957%, top4=0.942639946937561%, top5=0.968019962310791%, loss=0.12288984050124883, time=9.382826566696167s
Test epoch 2: top1=0.6656000018119812%, top2=0.8348999619483948%, top3=0.9080999493598938%, top4=0.9451999664306641%, top5=0.9685999751091003%, loss=0.11917177418507635, time=1.1009488105773926s
Train epoch 3: top1=0.717739999294281%, top2=0.8669399619102478%, top3=0.9289000034332275%, top4=0.9614599943161011%, top5=0.9793999791145325%, 

In [20]:
append_to_file(file_name='TRL_report.txt', text=f'########################################')

# TRL Method 1 just 3D tensors

In [35]:
import math
class TRL2(nn.Module):
    def __init__(self, input_shape, output_shape, rank, bias = False): # last rank for the output shape
          super(TRL2, self).__init__()
          self.G = nn.Parameter(torch.empty(rank), requires_grad=True)
          nn.init.kaiming_uniform_(self.G, a=math.sqrt(5))
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)
          self.fc4 = nn.Linear(rank[3], output_shape[0], bias = bias)

    def forward(self, x):
          G = self.fc4(self.G)
          x = self.fc3(x)
          x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = rearrange(x, 'b w h c -> b c h w')
          x = torch.einsum('b x y z , x y z d -> b d', x, G)

          return x

In [39]:
class CNN3(nn.Module):
    def __init__(self):
        super(CNN3, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.trl = TRL2(input_shape = (64,2,2), output_shape = (10,), rank=(10,1,1,10))
        # self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = x.reshape(x.size(0), 64,2,2)
        x = self.trl(x)
        return x


model3 = CNN3().to(device)


In [25]:
classifier3 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256),
    TRL2(input_shape = (64,4,4), output_shape = (10, ), rank=(10,1,1,10))
)

print(cp(classifier3))
append_to_file(file_name='TRL_report.txt', text=f'TRL method 1 classifier # parameters {cp(classifier3)}')

1049680


In [40]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model3.parameters())

In [41]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model3.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model3.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [42]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TRL_report.txt', text=f'TRL method 1 took {end_time - start_time} time')
append_to_file(file_name='TRL_report.txt', text=f'TRL method 1 had {string}')
append_to_file(file_name='TRL_report.txt', text=f'TRL method 1 last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

Train epoch 1: top1=0.5125399827957153%, top2=0.7167999744415283%, top3=0.8185399770736694%, top4=0.8833799958229065%, top5=0.9274399876594543%, loss=0.16717132340729238, time=8.939774751663208s
Test epoch 1: top1=0.6216999888420105%, top2=0.7985000014305115%, top3=0.8757999539375305%, top4=0.9273999929428101%, top5=0.9569000005722046%, loss=0.13638721650540828, time=1.0140666961669922s
Memory Usage  - Allocated: 65.25 MB, Reserved: 98.00 MB
Train epoch 2: top1=0.6696199774742126%, top2=0.8370199799537659%, top3=0.9077999591827393%, top4=0.9480999708175659%, top5=0.9713199734687805%, loss=0.11779714849755168, time=8.83468222618103s
Test epoch 2: top1=0.6836999654769897%, top2=0.8476999998092651%, top3=0.9145999550819397%, top4=0.953499972820282%, top5=0.9735999703407288%, loss=0.11238174115046859, time=1.0102667808532715s
Train epoch 3: top1=0.7312399744987488%, top2=0.8786199688911438%, top3=0.9363199472427368%, top4=0.9662399888038635%, top5=0.9825999736785889

In [43]:
append_to_file(file_name='TRL_report.txt', text=f'########################################')